# Cloud Torrent to Google Drive Downloader
This tool downloads Torrents or Magnet links directly in the cloud at gigabit speeds and uploads the completed files straight to your Google Drive, using 0% of your local internet data.

### Step 1: Connect your Google Drive
Run this cell below, click the authorization link, and sign in. (Or click the **Files 📁** icon in the left sidebar and click **Mount Drive**).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Install Downloader Engine
Run this cell to install **Aria2**, a high-performance command-line download utility that handles torrents and magnet links.

In [ ]:
!apt-get update
!apt-get install aria2 -y

### Step 3: Enter Torrent & Download
Paste your Magnet Link or Torrent URL on the right, choose your destination folder name, and click play to start downloading.

In [ ]:
#@title Torrent Downloader Form { display-mode: "form" }
MAGNET_OR_TORRENT_URL = "" #@param {type:"string"}
GDRIE_FOLDER_NAME = "Torrent_Downloads" #@param {type:"string"}

import os
import shutil
import subprocess
import time
import sys
import fcntl
import pty

LOCAL_DOWNLOAD_DIR = "/content/downloads"
TARGET_GDRIE_DIR = f"/content/drive/MyDrive/{GDRIE_FOLDER_NAME}"

if not MAGNET_OR_TORRENT_URL:
    print("\033[91m[ERROR] Please paste your Magnet link or Torrent URL into the form!\033[0m")
else:
    # Clean up previous local downloads if any
    if os.path.exists(LOCAL_DOWNLOAD_DIR):
        shutil.rmtree(LOCAL_DOWNLOAD_DIR)
    os.makedirs(LOCAL_DOWNLOAD_DIR, exist_ok=True)
    os.makedirs(TARGET_GDRIE_DIR, exist_ok=True)
    
    print(f"\033[94m[INFO] Starting torrent download to cloud server... Please wait.\033[0m")
    
    # Open a Pseudo-Terminal (PTY) master/slave pair to force interactive outputs (single-line updates)
    master, slave = pty.openpty()
    
    # Launch aria2c tied to the PTY
    cmd = ["aria2c", f"--dir={LOCAL_DOWNLOAD_DIR}", "--seed-time=0", "--disable-ipv6=true", MAGNET_OR_TORRENT_URL]
    process = subprocess.Popen(cmd, stdout=slave, stderr=slave, text=True, close_fds=True)
    
    # Close slave file descriptor in parent
    os.close(slave)
    
    # Set PTY master to non-blocking mode
    fl = fcntl.fcntl(master, fcntl.F_GETFL)
    fcntl.fcntl(master, fcntl.F_SETFL, fl | os.O_NONBLOCK)
    
    while True:
        # Try to read stream chunks from the master PTY
        try:
            chunk = os.read(master, 1024)
            if chunk:
                sys.stdout.write(chunk.decode('utf-8', errors='ignore'))
                sys.stdout.flush()
        except OSError:
            pass  # No new characters in buffer
            
        # Check if process exited naturally
        if process.poll() is not None:
            # Clean up remaining buffer output
            try:
                remaining = os.read(master, 4096)
                if remaining:
                    sys.stdout.write(remaining.decode('utf-8', errors='ignore'))
                    sys.stdout.flush()
            except:
                pass
            break
            
        time.sleep(0.1)
        
    # Clean up PTY file descriptor
    try:
        os.close(master)
    except:
        pass
        
    print(f"\n\033[94m[INFO] Download finished! Transferring files to Google Drive -> {GDRIE_FOLDER_NAME}...\033[0m")
    
    # Move downloaded files to Google Drive
    files_moved = 0
    for item in os.listdir(LOCAL_DOWNLOAD_DIR):
        source_path = os.path.join(LOCAL_DOWNLOAD_DIR, item)
        dest_path = os.path.join(TARGET_GDRIE_DIR, item)
        
        if os.path.exists(dest_path):
            if os.path.isdir(dest_path):
                shutil.rmtree(dest_path)
            else:
                os.remove(dest_path)
                
        shutil.move(source_path, dest_path)
        files_moved += 1
        print(f"[SUCCESS] Moved: {item}")
        
    print(f"\033[92m[SUCCESS] Successfully moved {files_moved} items to Google Drive!\033[0m")

### Step 4: Force Sync & Disconnect (Optional)
Run this cell once all your transfers are complete to instantly flush all cached data to Google Drive servers (so they appear on your phone/browser immediately) and safely unmount the connection.

In [ ]:
from google.colab import drive
print("[INFO] Flushing cache and disconnecting Drive...")
drive.flush_and_unmount()
print("[SUCCESS] Google Drive successfully synced and unmounted!")